# mini-vLLM prototype on Colab

Runtime → Change runtime type → **GPU** (T4 is fine for correctness checks; OPT-13B needs the cluster). Then run the cells in order.

Step 1 gets the code from GitHub. The repo is private, so create a fine-grained token (GitHub → Settings → Developer settings → Personal access tokens, read-only access to this repo) and paste it when asked. Rerun step 1 any time to `git pull` the latest code.

In [ ]:
import os, getpass
if not os.path.exists('/content/PagedAttention'):
    token = getpass.getpass('GitHub token (read access to tirthb101/PagedAttention): ')
    !git clone -q https://github.com/tirthb101/PagedAttention.git /content/PagedAttention
else:
    !git -C /content/PagedAttention pull -q
!git -C /content/PagedAttention log --oneline -1

In [ ]:
%cd /content/PagedAttention
!pip -q install transformers pytest
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Correctness tests (CPU + GPU-independent, ~15 s)

In [ ]:
!python -m pytest -q

## Real model demo

OPT-125m: 6 prompts, paged KV cache, checked token-by-token against HuggingFace.

In [ ]:
!python examples/demo.py --model facebook/opt-125m --max-tokens 32

## Force preemption with a tiny KV pool

8 blocks × 16 tokens can't hold all 6 requests at once, so the scheduler must preempt (recompute or swap) — the output must still match.

In [ ]:
!python examples/demo.py --model facebook/opt-125m --kv-blocks 8 --preemption recompute
!python examples/demo.py --model facebook/opt-125m --kv-blocks 8 --preemption swap

## Parallel sampling (prompt blocks shared, copy-on-write)

In [ ]:
!python examples/demo.py --model facebook/opt-125m --n 3 --temperature 0.8 --no-check

## Bigger model (fits a T4: 2.6 GB of weights)

In [ ]:
!python examples/demo.py --model facebook/opt-1.3b --max-tokens 64

## Triton kernel: correctness on the GPU, then speed (paper Fig. 18a)

In [ ]:
!python -m pytest -q tests/test_paged_triton.py
!python benchmarks/kernel_bench.py --model-dims opt-1.3b --contexts 64,128,256,512 --out results/e6_kernel_t4.csv

## Small serving experiment on the T4 (paged vs Orca, OPT-125m, synthetic ShareGPT)

Short 60 s traces at a few rates, KV pool capped at 0.5 GB so memory is the bottleneck. This is a smoke test of the benchmark pipeline; real results come from the cluster.

In [ ]:
for mm in ['paged', 'orca-oracle', 'orca-max']:
    !python benchmarks/run_serving.py --model facebook/opt-125m --trace synthetic-sharegpt --rates 1,2,4,8 --duration 60 --kv-cache-gb 0.5 --memory-manager {mm} --out results/t4_smoke.csv
!python benchmarks/plot.py results/t4_smoke.csv --out results/t4_smoke.png --title 'OPT-125m on T4, synthetic ShareGPT, 0.5 GB KV'

In [ ]:
from IPython.display import Image; Image('results/t4_smoke.png')